# Google Trends revisited: the API - Lane A (you prompt, the agent builds)

**SISMID 2026 - Day 2, 9:00.** Yesterday `pytrends` gave you 429s and numbers that moved
between pulls. Today we use the **Google Trends API**: same 0-100 signal, reliable and
reproducible.

First, in the **terminal**: `source scripts/unlock-gt-api-key.sh` (Google Trends passcode),
then restart the kernel so it sees `GT_API`.

**Quota: 1,000 requests/day for the whole room, health topics only.**


## Step 0: build the API helper

> *Write `gt_api_fetch(terms, geo, start, end)` that calls*
> *`https://www.googleapis.com/trends/v1beta/graph` with query parameters `key`, repeated*
> *`terms`, `restrictions.geo`, `restrictions.startDate`, `restrictions.endDate` (dates are*
> *YYYY-MM). Read the key from the GT_API environment variable, never hardcode it. Parse*
> *`lines[].points[]` into a tidy DataFrame: a date column plus one column per term.*


In [ ]:
# Paste the agent's helper here.
import os, time, json
import urllib.request, urllib.parse, urllib.error
import pandas as pd

ENDPOINT = "https://www.googleapis.com/trends/v1beta/graph"


def _norm(name: str) -> str:
    """Column-safe version of a term ('sintomas de dengue' -> 'sintomas_de_dengue')."""
    return name.strip().replace(" ", "_").replace("/", "_").lstrip("_")


def gt_api_fetch(terms, geo="US", start="2021-01", end=None, timeout=45):
    """Tidy DataFrame from the Google Trends API: a `date` column plus one
    column per term (0-100 normalized index).

    terms : str or list[str]  search terms or topic mids (e.g. '/m/0cycc')
    geo   : ISO-3166 code, country ('MX') or region ('US-GA')
    start / end : 'YYYY-MM' (end defaults to the current month)
    """
    key = os.getenv("GT_API")
    if not key:
        raise RuntimeError(
            "GT_API not set. Run `source scripts/unlock-gt-api-key.sh` in the "
            "terminal, then restart the kernel."
        )
    if isinstance(terms, str):
        terms = [terms]
    end = end or time.strftime("%Y-%m")

    params = [("key", key)] + [("terms", t) for t in terms] + [
        ("restrictions.geo", geo),
        ("restrictions.startDate", start),
        ("restrictions.endDate", end),
    ]
    url = ENDPOINT + "?" + urllib.parse.urlencode(params)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            data = json.loads(r.read())
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", "replace")[:300]
        raise RuntimeError(f"HTTP {e.code}: {body}") from e

    lines = data.get("lines") or []
    if not lines:
        raise RuntimeError(f"empty response for {terms} in {geo}")

    frames = []
    for line in lines:
        pts = line.get("points") or []
        frames.append(
            pd.DataFrame({
                "date": pd.to_datetime([p["date"] for p in pts]),
                _norm(line.get("term", "term")): [p["value"] for p in pts],
            }).set_index("date")
        )
    return pd.concat(frames, axis=1).reset_index().sort_values("date").reset_index(drop=True)


## Step 1: pull yesterday's dengue signal from the API

> *Pull 'dengue', 'sintomas de dengue' and 'mosquito' for Mexico (geo=MX) since 2021-07.*
> *Plot them, and report how many points you got and whether they are weekly or monthly.*
> *If the key is missing, fall back to ../data/gt_api_dengue_mx_cached.csv.*

**Your check:** does the last point land near today, and do the peaks match the dengue
season you saw yesterday?


In [ ]:
# Agent's pull:


## Step 2: prove it is reproducible

> *Pull the same series twice and check whether the values are identical.*

**Why it matters:** `pytrends` reads a sampled endpoint, so repeat pulls disagree. The API
is deterministic, which is what makes your analysis reproducible.


In [ ]:
# Agent's reproducibility check:


## Step 3: term vs topic across countries

> *For each of US, FR, IT and MX, pull the term 'flu' and the topic '/m/0cycc' (Influenza)*
> *in one call since 2022-01. Tabulate the max and mean of each per country, and draw a*
> *2x2 panel comparing the two series in each country.*

**What you should see:** the English term is nearly silent outside the US while the topic
tracks the flu season everywhere, because it covers *grippe*, *influenza* and *gripe*.
That is why the multi-country work later today uses topics.


In [ ]:
# Agent's term-vs-topic comparison:


## Step 4: reflect, then move on

- Same signal as yesterday, but no 429s and no drift between runs.
- Topics make one query work across languages.
- Unchanged: Google Trends is **attention**, not infection, and small places return zeros.

**Next:** streams that fail differently, Wikipedia (`01_...`) and wastewater (`02_...`).
